# Timed Auction Bid Fact Table — Gold Layer (`company_db.gold.bid_fact`)

This notebook builds and maintains the **Bid Fact** table in the Gold layer of the Company data warehouse. It captures every individual bid event in timed auctions, making it the most granular fact table in the warehouse — one row per unique bid attempt per user per lot per second.

---

## What This Notebook Does

1. Previews the Silver source data for inspection
2. Creates a staging view (`timed_auction`) that joins Silver bid records with the Gold Users Dimension to resolve surrogate keys
3. Initialises the `bid_fact` Delta table if it does not already exist
4. Merges staged bid records into the fact table (insert new bids, update existing ones)

---

## Source Tables

| Table | Layer | Description |
|---|---|---|
| `company_db.silver.timedauction` | Silver | Individual bid events from timed auctions — one row per bid action |
| `company_db.gold.users_dim` | Gold | User dimension — provides the `user_dim_key` surrogate FK |

## Target Table

`company_db.gold.bid_fact`  
Stored in Azure Data Lake Storage Gen2: `abfss://auction@company.dfs.core.windows.net/gold/bid_fact`

---

## MERGE Key

Records are matched on a **7-column composite key**: `auction_id + lot_id + user_id + bid_date + hour + minute + seconds`  
This uniquely identifies a single bid event down to the second, preventing duplicate rows while allowing multiple bids per user per lot across different times.

---

## Column Reference

| Column | Type | Description |
|---|---|---|
| `user_dim_key` | BIGINT | **Foreign key** → `company_db.gold.users_dim.user_dim_key` |
| `auction_id` | INTEGER | Auction identifier |
| `user_id` | INTEGER | Bidder’s source-system user ID |
| `bidder_id` | INTEGER | Platform-assigned bidder number for the auction |
| `lot_id` | INTEGER | Unique lot identifier |
| `lot_number` | INTEGER | Lot number within the auction catalogue |
| `current_bid` | DOUBLE | Bid amount at the time this record was captured |
| `max_bid` | DOUBLE | Maximum (proxy) bid placed by the bidder |
| `month` | INTEGER | Month number of the bid (1–12) |
| `month_day` | STRING | Day of the month of the bid |
| `hour` | INTEGER | Hour the bid was placed (0–23) |
| `minute` | INTEGER | Minute the bid was placed (0–59) |
| `seconds` | INTEGER | Second the bid was placed (0–59) |
| `week` | STRING | Week identifier of the bid |
| `bid_date` | DATE | Calendar date the bid was placed |
| `start_date` | DATE | Auction start date |
| `end_date` | DATE | Auction end date |
| `created_at` | TIMESTAMP | Timestamp when the row was first inserted |
| `updated_at` | TIMESTAMP | Timestamp of the last MERGE update |

In [ ]:
catalog = dbutils.widgets.get("catalog")
container = dbutils.widgets.get("container")
storage_account = dbutils.widgets.get("storage_account")

In [ ]:
%run "../setup/002_Environment_variables"

In [ ]:
from pyspark.sql import functions as F

---
## Step 1 · Silver Source Preview

Quick inspection of `company_db.silver.timedauction` before loading. Use this to:
- Confirm the Silver table has been refreshed with the latest bid data
- Check for unexpected nulls in key columns (`auction_id`, `lot_id`, `user_id`, `bid_date`)
- Verify the time columns (`hour`, `minute`, `seconds`) look reasonable before MERGE

> This cell is exploratory — it does not modify any data.

In [ ]:
spark.sql(f"""
CREATE VIEW IF NOT EXISTS {catalog}.gold.timed_auction AS
(SELECT
    ud.user_dim_key,
    td.auction_id,
    td.user_id,
    td.bidder_id,
    td.lot_id,
    td.lot_number,
    td.current_bid,
    td.max_bid,
    td.month,
    td.month_day,
    td.hour,
    td.minute,
    td.seconds,
    td.week,
    td.bid_date,    
    td.start_date,
    td.end_date
FROM
{catalog}.silver.timedauction as td
LEFT JOIN {catalog}.gold.users_dim as ud
ON td.user_id = ud.user_id)""")

---
## Step 2 · Initialize Gold Fact Table

Creates `company_db.gold.bid_fact` as a **Delta table** if it does not already exist. On subsequent runs this is a no-op — no existing data is affected.

Granularity: **one row per unique bid event**, identified by `auction_id + lot_id + user_id + bid_date + hour + minute + seconds`. This means the table can hold multiple bids from the same user on the same lot as long as they occurred at different timestamps.

The table is backed by ADLS Gen2 at `gold/bid_fact` and supports full ACID transactions (MERGE, time travel, schema evolution).

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.gold.bid_fact
(
  user_dim_key BIGINT,
  auction_id INTEGER,
  user_id INTEGER,
  bidder_id INTEGER,
  lot_id INTEGER,
  lot_number INTEGER,
  current_bid DOUBLE,
  max_bid DOUBLE,
  month INTEGER,
  month_day STRING,
  hour INTEGER,
  minute INTEGER,
  seconds INTEGER,
  week STRING,
  bid_date DATE,
  start_date DATE,
  end_date DATE,
  created_at TIMESTAMP,
  updated_at TIMESTAMP
)
USING DELTA
LOCATION 'abfss://{container}@{storage_account}.dfs.core.windows.net/gold/bid_fact';""")

---
## Step 3 · MERGE into Gold Fact Table

Applies staged bid records from `company_db.gold.timed_auction` into `bid_fact` using the 7-column composite key.

| Condition | Action | Fields Updated |
|---|---|---|
| MATCHED | **UPDATE** bid amounts + `updated_at = current_timestamp()` | `current_bid`, `max_bid`, `updated_at` |
| NOT MATCHED | **INSERT** full row with `created_at = current_timestamp()`, `updated_at = current_timestamp()` | All columns |

**Why only `current_bid` and `max_bid` are updated on MATCH:**  
The time-based composite key means a matched row is the same bid event being re-processed (e.g. a Silver table refresh). Only the bid amounts are updated since all other fields (lot, dates, user) are already correct.

In [ ]:
spark.sql(f"""
MERGE INTO {catalog}.gold.bid_fact AS target
USING {catalog}.gold.timed_auction AS source
ON target.auction_id = source.auction_id
   AND target.lot_id = source.lot_id
   AND target.user_id = source.user_id
   AND target.bid_date = source.bid_date
   AND target.hour = source.hour
   AND target.minute = source.minute
   AND target.seconds = source.seconds
WHEN MATCHED THEN
UPDATE SET
  target.current_bid = source.current_bid,
  target.max_bid = source.max_bid,
  target.updated_at = current_timestamp()
WHEN NOT MATCHED THEN
INSERT (
  user_dim_key, auction_id, user_id, bidder_id, lot_id, lot_number,
  current_bid, max_bid, month, month_day, hour, minute, seconds, week,
  bid_date, start_date, end_date, created_at, updated_at
)
VALUES (
  source.user_dim_key, source.auction_id, source.user_id, source.bidder_id,
  source.lot_id, source.lot_number, source.current_bid, source.max_bid,
  source.month, source.month_day, source.hour, source.minute, source.seconds,
  source.week, source.bid_date, source.start_date, source.end_date,
  current_timestamp(), current_timestamp()
)""")

---
## Step 4 · Verification

Queries `company_db.gold.bid_fact` to validate the MERGE output. Spot-check that:
- Row count is consistent with expected bid volume
- `user_dim_key` is non-null for known registered bidders
- No duplicate rows exist on the composite key
- `current_bid` and `max_bid` are positive for all rows
- `created_at` and `updated_at` are populated

In [ ]:

# Verify MERGE output: row counts, bid integrity, and duplicate check
spark.sql(f"""
SELECT
  COUNT(*)                                                      AS total_bids,
  COUNT(DISTINCT auction_id)                                    AS unique_auctions,
  COUNT(DISTINCT user_id)                                       AS unique_bidders,
  COUNT(DISTINCT lot_id)                                        AS unique_lots,
  SUM(CASE WHEN user_dim_key IS NULL THEN 1 END)                AS missing_dim_keys,
  SUM(CASE WHEN current_bid <= 0 OR max_bid <= 0 THEN 1 END)   AS invalid_bid_amounts,
  ROUND(AVG(current_bid), 2)                                    AS avg_current_bid,
  MAX(updated_at)                                               AS latest_updated
FROM {catalog}.gold.bid_fact;""")

In [ ]:
# spark.table('company_db.gold.timed_auction').filter(F.col("user_dim_key").isNull()).display()

In [ ]:
# %sql
# select * from company_db.silver.timedauction
# where user_id is null


In [ ]:
%sql
-- DESCRIBE HISTORY company_db.gold.bid_fact;
-- RESTORE TABLE company_db.gold.bid_fact TO VERSION AS OF 1;